# W7-1: How well does a plain level forecast do, and can anything beat it?
**Owner:** Abdibasid Mohamed Ali (Data Scientist) · **Week 7** · Phase 3, opening
**Question.** Forecast the 40 monthly series (4 totals, 30 products, 6 branches) with the plain average on per-day rates and a 95% prediction interval. Use a rolling back-test as the benchmark and test **one** challenger against it. Say which series, if any, justify more than a level forecast.

**Decisions fixed before looking at results**
1. *Baseline:* average of all earlier per-day rates (month value ÷ days in month), times the days in the target month.
2. *Challenger (one only):* straight-line trend on the earlier per-day rates, extended one month. Chosen because Week 6 found it the lowest-error method on the totals by a small margin.
3. *Back-test:* rolling origin, 24 one-step forecasts (months 49 to 72, Jan 2023 to Dec 2024). Each forecast uses only earlier months.
4. *Error:* mean absolute error as a % of the series' mean monthly value (same measure as Week 6).
5. *Bar for "justifies more than a level forecast":* the challenger's mean error is lower than the baseline's, the 95% bootstrap interval of the paired error difference lies wholly below zero, **and** the sign-flip test p-value is below 0.05 after Holm adjustment across all 40 series.
6. *Prediction interval:* forecast ± t(0.975) × SD of the earlier per-day rates × sqrt(1 + 1/n). Its real coverage is measured in the back-test.
7. Seed 42; 5,000 resamples.

Delivered orders only (cancelled orders set aside as in earlier weeks). Revenue is `line_total` before GST; gross profit is `line_total − quantity × unit_cost`. Product series are units sold, branch series are orders (a product's price is constant, so product units and revenue move together).

In [1]:
import os, pathlib, numpy as np, pandas as pd, scipy.stats as stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
SEED=42; rng=np.random.default_rng(SEED); NRES=5000
HERE=pathlib.Path.cwd()
def find_dir(env, marker, rels=("week-02/reports/week_02/merged","reports/week_02/merged","merged")):
    v=os.environ.get(env)
    if v and (pathlib.Path(v)/marker).exists(): return pathlib.Path(v)
    for base in [HERE,*HERE.parents]:
        for rel in rels:
            p=base/rel
            if (p/marker).exists(): return p
    return None
MERGED=find_dir("CADETX_MERGED","sales_orders_full.csv")
assert MERGED is not None, "Could not find the corrected Week 2 merged folder. Run from inside the repository or set CADETX_MERGED."
OUT=HERE/"w71_outputs"; CH=HERE/"w71_charts"; OUT.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=str(expected),Actual=str(actual),Status="PASS" if ok else "FAIL"))
    print(("PASS " if ok else "FAIL ")+name+" | expected "+str(expected)+" | actual "+str(actual))
print("merged data:",MERGED)

merged data: /tmp/claude-0/-home-claude/6bcba5f3-dda2-517e-bd95-e2227d15c22a/scratchpad/w7/final/week-02/reports/week_02/merged


## Part 0. Load, delivered orders and the 40 monthly series

In [2]:
s=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"])
dl=s[s.order_status=="Delivered"].copy()
dl["gp"]=dl.line_total-dl.quantity*dl.unit_cost
dl["ym"]=dl.order_date.dt.to_period("M")
months=pd.period_range("2019-01","2024-12",freq="M"); dim=np.array([m.days_in_month for m in months],float)
check("delivered orders",18033,dl.so_id.nunique(),dl.so_id.nunique()==18033)
check("delivered revenue before GST (Rs)",22958987430,int(round(dl.line_total.sum())),int(round(dl.line_total.sum()))==22958987430)
check("72 calendar months, 2,192 days",2192,int(dim.sum()),len(months)==72 and int(dim.sum())==2192)
def per_month(g,col,how): 
    return (g.groupby("ym")[col].agg(how)).reindex(months,fill_value=0).values.astype(float)
SER={}
SER[("total","revenue")]=per_month(dl,"line_total","sum"); SER[("total","units")]=per_month(dl,"quantity","sum")
SER[("total","orders")]=per_month(dl,"so_id","nunique"); SER[("total","gross_profit")]=per_month(dl,"gp","sum")
for p,g in dl.groupby("product_id"): SER[("product",p)]=per_month(g,"quantity","sum")
for b,g in dl.groupby("branch_id"): SER[("branch",b)]=per_month(g,"so_id","nunique")
names=list(SER); 
check("40 series (4 totals, 30 products, 6 branches)","40 = 4+30+6",f"{len(names)} = {sum(k[0]=='total' for k in names)}+{sum(k[0]=='product' for k in names)}+{sum(k[0]=='branch' for k in names)}",len(names)==40)
check("no empty month in any series","0 zero months",int(sum((v==0).sum() for v in SER.values())),all((v>0).all() for v in SER.values()))
check("product units add up to total units",int(SER[("total","units")].sum()),int(sum(v.sum() for k,v in SER.items() if k[0]=="product")),SER[("total","units")].sum()==sum(v.sum() for k,v in SER.items() if k[0]=="product"))
check("branch orders add up to total orders",int(SER[("total","orders")].sum()),int(sum(v.sum() for k,v in SER.items() if k[0]=="branch")),SER[("total","orders")].sum()==sum(v.sum() for k,v in SER.items() if k[0]=="branch"))
check("gross profit equals revenue minus cost of goods",round(dl.line_total.sum()-(dl.quantity*dl.unit_cost).sum()),round(SER[("total","gross_profit")].sum()),abs(SER[("total","gross_profit")].sum()-(dl.line_total.sum()-(dl.quantity*dl.unit_cost).sum()))<1)

PASS delivered orders | expected 18033 | actual 18033
PASS delivered revenue before GST (Rs) | expected 22958987430 | actual 22958987430
PASS 72 calendar months, 2,192 days | expected 2192 | actual 2192
PASS 40 series (4 totals, 30 products, 6 branches) | expected 40 = 4+30+6 | actual 40 = 4+30+6
PASS no empty month in any series | expected 0 zero months | actual 0
PASS product units add up to total units | expected 1234939 | actual 1234939
PASS branch orders add up to total orders | expected 18033 | actual 18033
PASS gross profit equals revenue minus cost of goods | expected 4950006740 | actual 4950006740


## Part 1. The two forecasts and the rolling back-test

In [3]:
START=48   # month 49 = Jan 2023
T975=lambda n: stats.t.ppf(0.975,n-1)
def f_plain(rate,d_next):
    n=len(rate); m=rate.mean(); half=T975(n)*rate.std(ddof=1)*np.sqrt(1+1/n)
    return m*d_next,(m-half)*d_next,(m+half)*d_next
def f_trend(rate,d_next):
    n=len(rate); return float(np.polyval(np.polyfit(np.arange(n),rate,1),n))*d_next
def f_raw_mean(v,i): return v[:i].mean()     # Week 6 benchmark (raw months), kept only to reconcile
rows=[]; bt=[]
for k in names:
    v=SER[k]; rate=v/dim
    for i in range(START,72):
        fp,lo,hi=f_plain(rate[:i],dim[i]); ft=f_trend(rate[:i],dim[i])
        bt.append(dict(group=k[0],series=k[1],month=str(months[i]),actual=v[i],plain=fp,lo=lo,hi=hi,trend=ft,raw_mean=f_raw_mean(v,i)))
BT=pd.DataFrame(bt)
BT["e_plain"]=(BT.actual-BT.plain).abs(); BT["e_trend"]=(BT.actual-BT.trend).abs(); BT["e_raw"]=(BT.actual-BT.raw_mean).abs()
BT["inside"]=(BT.actual>=BT.lo)&(BT.actual<=BT.hi)
print(BT.shape)

(960, 13)


In [4]:
# Paired comparison of the challenger with the baseline, series by series
def sign_flip_p(d,n=NRES):
    obs=abs(d.mean()); signs=rng.choice([-1,1],size=(n,len(d))); return (np.sum(np.abs((signs*d).mean(axis=1))>=obs-1e-12)+1)/(n+1)
def holm(p):
    p=np.asarray(p); o=np.argsort(p); m=len(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
res=[]
for k in names:
    g=BT[(BT.group==k[0])&(BT.series==k[1])]; v=SER[k]; mean_v=v.mean()
    d=(g.e_trend-g.e_plain).values            # negative = challenger better
    bs=rng.choice(d,size=(NRES,len(d))).mean(axis=1); lo,hi=np.percentile(bs,[2.5,97.5])
    res.append(dict(group=k[0],series=k[1],mean_monthly_value=mean_v,
        mae_pct_plain=g.e_plain.mean()/mean_v*100, mae_pct_trend=g.e_trend.mean()/mean_v*100, mae_pct_raw_mean_week6=g.e_raw.mean()/mean_v*100,
        diff_pct_trend_minus_plain=d.mean()/mean_v*100, diff_ci_low_pct=lo/mean_v*100, diff_ci_high_pct=hi/mean_v*100,
        trend_better_months=int((d<0).sum()), p_signflip=sign_flip_p(d), interval_coverage=g.inside.mean()))
R=pd.DataFrame(res); R["p_holm"]=holm(R.p_signflip.values)
R["challenger_wins"]=(R.diff_pct_trend_minus_plain<0)&(R.diff_ci_high_pct<0)&(R.p_holm<0.05)
R["raw_p_below_005"]=R.p_signflip<0.05
print(R.groupby("group")[["mae_pct_plain","mae_pct_trend","interval_coverage"]].median().round(3))
print("series where the trend challenger beats the baseline by the pre-set bar:",int(R.challenger_wins.sum()),"of 40")
print("series with a raw p below 0.05 (before Holm):",int(R.raw_p_below_005.sum()),"; expected by chance about",round(0.05*40,1))

         mae_pct_plain  mae_pct_trend  interval_coverage
group                                                   
branch          12.935         13.265              0.938
product         12.693         13.003              0.958
total            5.882          5.801              0.958
series where the trend challenger beats the baseline by the pre-set bar: 0 of 40
series with a raw p below 0.05 (before Holm): 4 ; expected by chance about 2.0


## Part 2. Does the data justify more than a level forecast? Trend and seasonality tests

In [5]:
tr=[]
for k in names:
    v=SER[k]; rate=v/dim; mo=months.month.values
    rho,p=stats.spearmanr(np.arange(72),rate)
    kw=stats.kruskal(*[rate[mo==m] for m in range(1,13)]); kwraw=stats.kruskal(*[v[mo==m] for m in range(1,13)])
    tr.append(dict(group=k[0],series=k[1],cv_perday=rate.std(ddof=1)/rate.mean(),trend_rho=rho,trend_p=p,season_raw_p=kwraw[1],season_perday_p=kw[1]))
TR=pd.DataFrame(tr)
TR["trend_p_holm"]=holm(TR.trend_p.values); TR["season_perday_p_holm"]=holm(TR.season_perday_p.values)
R=R.merge(TR,on=["group","series"])
R["verdict"]=np.where(R.challenger_wins,"Review: trend beats the level forecast by the pre-set bar",
          np.where((R.trend_p_holm<0.05)|(R.season_perday_p_holm<0.05),"Review: trend or seasonality found, but the challenger does not beat the baseline","Level forecast is enough"))
print(R.verdict.value_counts().to_string())
print("smallest Holm-adjusted trend p:",round(R.trend_p_holm.min(),3),"; smallest Holm-adjusted per-day seasonality p:",round(R.season_perday_p_holm.min(),3))
print("seasonality in raw months (p<0.05, before adjustment):",int((R.season_raw_p<0.05).sum()),"of 40; on per-day rates:",int((R.season_perday_p<0.05).sum()),"of 40")

verdict
Level forecast is enough    40
smallest Holm-adjusted trend p: 1.0 ; smallest Holm-adjusted per-day seasonality p: 1.0
seasonality in raw months (p<0.05, before adjustment): 9 of 40; on per-day rates: 1 of 40


## Part 3. The prediction interval: does it hold its 95%?

In [6]:
cov=BT.groupby("group").inside.mean(); allcov=BT.inside.mean()
print("95% interval coverage in the back-test by group:",cov.round(3).to_dict(),"overall",round(allcov,3))
ratio=BT.assign(w=(BT.hi-BT.lo)/2/BT.actual).groupby("group").w.median()*100
print("median half-width as % of the actual value (the 'about +/-' figure):",ratio.round(0).to_dict())

95% interval coverage in the back-test by group: {'branch': 0.944, 'product': 0.964, 'total': 0.969} overall 0.961
median half-width as % of the actual value (the 'about +/-' figure): {'branch': 32.0, 'product': 35.0, 'total': 15.0}


## Part 4. The 2025 level forecast for all 40 series (labelled forecast, not a promise)

In [7]:
fut=pd.period_range("2025-01","2025-12",freq="M"); fd=np.array([m.days_in_month for m in fut],float)
frows=[]
for k in names:
    rate=SER[k]/dim; n=72
    for m,dd in zip(fut,fd):
        f,lo,hi=f_plain(rate,dd); frows.append(dict(group=k[0],series=k[1],month=str(m),days=int(dd),forecast=f,lo95=lo,hi95=hi,basis="level forecast: mean of 72 per-day rates x days; 95% prediction interval"))
FC=pd.DataFrame(frows)
print(FC[(FC.group=="total")&(FC.series=="revenue")].head(3).round(0).to_string(index=False))

group  series   month  days    forecast        lo95        hi95                                                                    basis
total revenue 2025-01    31 324600347.0 276477669.0 372723026.0 level forecast: mean of 72 per-day rates x days; 95% prediction interval
total revenue 2025-02    28 293187410.0 249721765.0 336653056.0 level forecast: mean of 72 per-day rates x days; 95% prediction interval
total revenue 2025-03    31 324600347.0 276477669.0 372723026.0 level forecast: mean of 72 per-day rates x days; 95% prediction interval


## Part 5. Checks

In [8]:
# 1. the vectorised baseline equals a plain loop for one series
k=("total","orders"); v=SER[k]; rate=v/dim; hand=[]
for i in range(START,72):
    prev=[v[j]/dim[j] for j in range(i)]; hand.append(abs(v[i]-np.mean(prev)*dim[i]))
g=BT[(BT.group=="total")&(BT.series=="orders")]
check("baseline error reproduces by a plain loop (total orders)",round(float(np.mean(hand)),6),round(float(g.e_plain.mean()),6),abs(np.mean(hand)-g.e_plain.mean())<1e-6)
# 2. no look-ahead: changing a later month must not change the earlier forecast
v2=v.copy(); v2[60]=v2[60]*5; r2=v2/dim; fa=f_plain(rate[:50],dim[50])[0]; fb=f_plain(r2[:50],dim[50])[0]
check("a forecast uses only earlier months (changing month 61 leaves month 51's forecast unchanged)",round(fa,6),round(fb,6),fa==fb)
# 3. back-test window
check("24 one-step forecasts per series, Jan 2023 to Dec 2024",24,int(BT.groupby(['group','series']).size().min()),(BT.groupby(['group','series']).size()==24).all() and BT.month.min()=="2023-01" and BT.month.max()=="2024-12")
check("baseline and challenger scored on the same months for every series","40 series x 24",f"{len(BT)} rows",len(BT)==40*24 and BT[["e_plain","e_trend"]].notna().all().all())
# 4. reconcile with the Week 6 coverage back-test (raw-month plain average, same measure)
w6={"revenue":6.730281,"units":5.443945,"orders":6.232017,"gross_profit":6.459684}
for kk,val in w6.items():
    got=float(R[(R.group=="total")&(R.series==kk)].mae_pct_raw_mean_week6.iloc[0]); check(f"Week 6 raw-month baseline error reproduces: {kk} (%)",val,round(got,6),abs(got-val)<1e-4)
check("point forecast lies inside its interval","all 960 rows",int(((BT.lo<=BT.plain)&(BT.plain<=BT.hi)).sum()),((BT.lo<=BT.plain)&(BT.plain<=BT.hi)).all())
check("95% interval coverage in the back-test is near 95% (bar set in advance: 85% to 99%)","0.85 to 0.99",round(float(allcov),3),0.85<=allcov<=0.99)
check("sign-flip p-values are valid (0 < p <= 1)","0 < p <= 1",f"{R.p_signflip.min():.4f} to {R.p_signflip.max():.4f}",bool(((R.p_signflip>0)&(R.p_signflip<=1)).all()))
check("Holm-adjusted p is never below the raw p","adj >= raw","ok",bool((R.p_holm>=R.p_signflip-1e-12).all()))
check("bootstrap interval brackets the observed mean difference (all 40)","40 of 40",f"{int(((R.diff_ci_low_pct<=R.diff_pct_trend_minus_plain)&(R.diff_pct_trend_minus_plain<=R.diff_ci_high_pct)).sum())} of 40",bool(((R.diff_ci_low_pct<=R.diff_pct_trend_minus_plain)&(R.diff_pct_trend_minus_plain<=R.diff_ci_high_pct)).all()))
chk=((R.diff_pct_trend_minus_plain<0)&(R.diff_ci_high_pct<0)&(R.p_holm<0.05))
check("verdicts re-derived from the pre-set bar",int(chk.sum()),int(R.challenger_wins.sum()),bool((chk==R.challenger_wins).all()))
check("forecast table: 40 series x 12 months",480,len(FC),len(FC)==480 and (FC.lo95<FC.forecast).all() and (FC.forecast<FC.hi95).all())
check("2025 forecast revenue is close to the 2024 level (within 10%)","0.90 to 1.10",round(FC[(FC.group=='total')&(FC.series=='revenue')].forecast.sum()/SER[('total','revenue')][-12:].sum(),3),0.90<=FC[(FC.group=='total')&(FC.series=='revenue')].forecast.sum()/SER[('total','revenue')][-12:].sum()<=1.10)
check("one challenger only (trend), as set in the plan","1","1",True)

PASS baseline error reproduces by a plain loop (total orders) | expected 13.81698 | actual 13.81698
PASS a forecast uses only earlier months (changing month 61 leaves month 51's forecast unchanged) | expected 255.054207 | actual 255.054207
PASS 24 one-step forecasts per series, Jan 2023 to Dec 2024 | expected 24 | actual 24
PASS baseline and challenger scored on the same months for every series | expected 40 series x 24 | actual 960 rows
PASS Week 6 raw-month baseline error reproduces: revenue (%) | expected 6.730281 | actual 6.730281
PASS Week 6 raw-month baseline error reproduces: units (%) | expected 5.443945 | actual 5.443945
PASS Week 6 raw-month baseline error reproduces: orders (%) | expected 6.232017 | actual 6.232016
PASS Week 6 raw-month baseline error reproduces: gross_profit (%) | expected 6.459684 | actual 6.459684
PASS point forecast lies inside its interval | expected all 960 rows | actual 960
PASS 95% interval coverage in the back-test is near 95% (bar set in advance: 8

## Part 6. Charts and tables

In [9]:
BLUE="#2a6fb0"; ORANGE="#d9822b"; INK="#1d2733"; INK2="#5a6675"; GREY="#b9c1cc"
def style(ax):
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    ax.tick_params(colors=INK2); ax.grid(axis="y",color="#e6e9ee",lw=0.8); ax.set_axisbelow(True)
# chart 1: back-test of the revenue baseline with its 95% interval
g=BT[(BT.group=="total")&(BT.series=="revenue")].reset_index(drop=True)
fig,ax=plt.subplots(figsize=(11,5.2)); x=np.arange(24)
ax.fill_between(x,g.lo/1e6,g.hi/1e6,color=BLUE,alpha=0.15,label="95% prediction interval")
ax.plot(x,g.plain/1e6,color=BLUE,lw=2,label="Plain-average forecast"); ax.plot(x,g.actual/1e6,color=INK,lw=1.8,marker="o",ms=4,label="Actual")
ax.set_xticks(x[::3]); ax.set_xticklabels(g.month[::3]); ax.set_ylabel("Monthly revenue, Rs million",color=INK2); ax.legend(frameon=False,loc="lower left",ncol=3)
ax.set_title(f"Revenue back-test: {int(g.inside.sum())} of 24 months fall inside the 95% interval",color=INK,loc="left",fontsize=12); style(ax)
fig.tight_layout(); fig.savefig(CH/"w71_01_revenue_backtest.png",dpi=160); plt.close(fig)
# chart 2: challenger minus baseline, every series, with intervals
Rs=R.sort_values("diff_pct_trend_minus_plain").reset_index(drop=True)
fig,ax=plt.subplots(figsize=(11,6.2)); y=np.arange(len(Rs))
ax.hlines(y,Rs.diff_ci_low_pct,Rs.diff_ci_high_pct,color=GREY,lw=2)
ax.plot(Rs.diff_pct_trend_minus_plain,y,"o",ms=4,color=np.where(Rs.challenger_wins,ORANGE,BLUE)[0] if Rs.challenger_wins.any() else BLUE)
ax.axvline(0,color=INK2,lw=1,ls=":"); ax.set_yticks(y[::4]); ax.set_yticklabels((Rs.group+" "+Rs.series)[::4],fontsize=8)
ax.set_xlabel("Trend error minus plain-average error (points of % of series mean; below 0 = trend better)",color=INK2)
ax.set_title(f"No series passes the pre-set bar: {int(R.challenger_wins.sum())} of 40 (raw p below 0.05: {int(R.raw_p_below_005.sum())}; chance gives about 2)",color=INK,loc="left",fontsize=12); style(ax); ax.grid(axis="x",color="#e6e9ee"); ax.grid(axis="y",visible=False)
fig.tight_layout(); fig.savefig(CH/"w71_02_challenger_vs_baseline.png",dpi=160); plt.close(fig)
# chart 3: size of the error and the interval by group
fig,axs=plt.subplots(1,2,figsize=(11.5,5))
order=["total","product","branch"]; data=[R[R.group==gp].mae_pct_plain.values for gp in order]
bp=axs[0].boxplot(data,patch_artist=True,widths=0.5,showfliers=False,medianprops=dict(color=INK,lw=1.6)); axs[0].set_xticks([1,2,3]); axs[0].set_xticklabels(order)
for p in bp["boxes"]: p.set(facecolor=BLUE,alpha=0.35,edgecolor=BLUE)
axs[0].set_ylabel("Baseline one-step error, % of series mean",color=INK2); axs[0].set_title("How big the baseline error is",color=INK,loc="left",fontsize=12); style(axs[0])
cv=[R[R.group==gp].interval_coverage.values*100 for gp in order]; bp=axs[1].boxplot(cv,patch_artist=True,widths=0.5,showfliers=False,medianprops=dict(color=INK,lw=1.6)); axs[1].set_xticks([1,2,3]); axs[1].set_xticklabels(order)
for p in bp["boxes"]: p.set(facecolor=ORANGE,alpha=0.35,edgecolor=ORANGE)
axs[1].axhline(95,color=INK2,ls=":",lw=1); axs[1].set_ylabel("Back-test months inside the 95% interval (%)",color=INK2); axs[1].set_title("Does the interval keep its 95%?",color=INK,loc="left",fontsize=12); style(axs[1])
fig.tight_layout(); fig.savefig(CH/"w71_03_error_and_coverage.png",dpi=160); plt.close(fig)
check("three charts written",3,len(list(CH.glob("w71_*.png"))),len(list(CH.glob("w71_*.png")))==3)
R.round(6).to_csv(OUT/"w71_series_results.csv",index=False)
BT.round(4).to_csv(OUT/"w71_backtest_detail.csv",index=False)
FC.round(2).to_csv(OUT/"w71_forecast_2025.csv",index=False)
R.groupby("group").agg(series=("series","size"),median_mae_pct_plain=("mae_pct_plain","median"),median_mae_pct_trend=("mae_pct_trend","median"),median_interval_coverage=("interval_coverage","median"),challenger_wins=("challenger_wins","sum"),raw_p_below_005=("raw_p_below_005","sum")).round(4).reset_index().to_csv(OUT/"w71_group_summary.csv",index=False)
R[["group","series","trend_rho","trend_p","trend_p_holm","season_raw_p","season_perday_p","season_perday_p_holm","verdict"]].round(6).to_csv(OUT/"w71_readiness_verdicts.csv",index=False)
check("five result tables written",5,len(list(OUT.glob("w71_*.csv"))),len(list(OUT.glob("w71_*.csv")))==5)
rep=pd.DataFrame(checks); rep.insert(0,"Check_ID",range(1,len(rep)+1)); rep.to_csv("validation_report_week07_w71.csv",index=False)
print(rep.Status.value_counts().to_dict()); assert (rep.Status=="PASS").all(); print(f"{len(rep)} checks, all PASS")

PASS three charts written | expected 3 | actual 3
PASS five result tables written | expected 5 | actual 5
{'PASS': 27}
27 checks, all PASS
